# 08 — Read-only cloud diagnostics
Inspect the exact object metadata returned by gcloud. No objects are overwritten.

In [ ]:
import json, subprocess
from IPython.display import display
from pathlib import Path
root = Path.cwd().resolve()
if root.name == "notebooks": root = root.parent
import sys
EXPECTED_RUN_ID = "20261012-full-01"
runtime = json.loads(
    (root / "artifacts/execution/runs" / EXPECTED_RUN_ID / "runtime.json").read_text()
)
sys.path.insert(0, str(root / "src"))
from rainfall.billing import validate_runtime_credit_gate
billing = json.loads(subprocess.run(
    ["gcloud", "billing", "projects", "describe", runtime["project_id"], "--format=json"],
    capture_output=True, text=True, check=True
).stdout)
display(validate_runtime_credit_gate(
    runtime, project_id=runtime["project_id"], billing_info=billing
))
manifest = json.loads((root / "manifests/input_manifest.json").read_text())
record = manifest["files"][0]
uri = "gs://sg-rainfall-dsa5208-261011-data/" + record["relative_path"]
result = subprocess.run(["gcloud", "storage", "objects", "describe", uri, "--format=json"], capture_output=True, text=True, check=True)
info = json.loads(result.stdout)
print(json.dumps(info, indent=2))
print("Expected SHA256:",record["sha256"])


In [ ]:
result = subprocess.run(["gcloud", "dataproc", "clusters", "describe", "dsa5208-rainfall", "--project=sg-rainfall-dsa5208-261011", "--region=asia-southeast1", "--format=json"], capture_output=True, text=True, check=True)
cluster = json.loads(result.stdout)
print(json.dumps({"state": cluster["status"]["state"], "status_detail": cluster["status"].get("detail"), "master": cluster["config"]["masterConfig"], "workers": cluster["config"]["workerConfig"], "image": cluster["config"]["softwareConfig"]["imageVersion"], "expiry": cluster["config"].get("lifecycleConfig")},indent=2))
(root / "artifacts/execution/runs" / EXPECTED_RUN_ID / "cluster_snapshot.json").write_text(json.dumps(cluster,indent=2))


In [ ]:
import re
zone = cluster["config"]["gceClusterConfig"]["zoneUri"].rsplit("/",1)[-1]
master = cluster["config"]["masterConfig"]["instanceNames"][0]
command = f"gcloud storage objects describe {runtime['source_bundle_uri']} --format='value(name)'"
result = subprocess.run(
    ["gcloud", "compute", "ssh", master, f"--zone={zone}",
     "--project=sg-rainfall-dsa5208-261011", "--tunnel-through-iap", "--quiet",
     f"--command={command}"],
    capture_output=True, text=True,
)
print("Remote read exit code:", result.returncode)
print(re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr))
print(result.stdout)
if result.returncode:
    raise RuntimeError("Dataproc VM could not read the versioned source bundle.")
